#### Lib Imports

In [26]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import mlflow as mf

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import mlflow.xgboost


In [2]:
mf.set_tracking_uri("http://127.0.0.1:5000")

In [3]:
mf.set_experiment("HousingPricePrediction")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1777502087548, experiment_id='1', last_update_time=1777502087548, lifecycle_stage='active', name='HousingPricePrediction', tags={}, trace_location=None, workspace='default'>

In [4]:
mf.sklearn.autolog()

#### Data Load

In [5]:
df = pd.read_csv("../data/raw/housing.csv")

In [6]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


#### Feature Engneering

In [7]:
# I would like to understand if the null values mean they are studio flats...
df["rooms_per_household"] = df["total_rooms"] / df["households"]

In [8]:
mapping = {
    'INLAND': 0,
    '<1H OCEAN': 1,
    'NEAR OCEAN': 2,
    'NEAR BAY': 3,
    'ISLAND': 4
}
df["ocean_proximity_c"] = df["ocean_proximity"].map(mapping)

In [9]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,rooms_per_household,ocean_proximity_c
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY,6.984127,3
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,6.238137,3
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,8.288136,3
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,5.817352,3
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,6.281853,3


In [10]:
df.isnull().sum()

longitude                0
latitude                 0
housing_median_age       0
total_rooms              0
total_bedrooms         207
population               0
households               0
median_income            0
median_house_value       0
ocean_proximity          0
rooms_per_household      0
ocean_proximity_c        0
dtype: int64

In [11]:
df["total_bedrooms"] = df["total_bedrooms"].fillna(df["total_bedrooms"].median())

In [12]:
print(df["total_bedrooms"].isnull().sum())

0


In [13]:
df.columns

Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value', 'ocean_proximity', 'rooms_per_household',
       'ocean_proximity_c'],
      dtype='object')

In [14]:
features = ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income','rooms_per_household',
       'ocean_proximity_c']

#### Models

In [15]:
X = df[features]
y = df["median_house_value"]

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [17]:
# Satandardization : 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

#### MLFlow & Experiment

In [29]:
# Autologs
mf.sklearn.autolog(silent=True)
mlflow.xgboost.autolog(silent=True)

# Main (Parent) Run:
with mf.start_run(run_name="Housing_Model_Comparison_v0"):
    # 1. Ortak Tagler ve Scaler Kaydı (Sadece 1 kere)
    mf.set_tag("source_file", "housing.csv")
    mf.set_tag("processed_data", "housing_proc.csv")
    mf.set_tag("dataset_version", "v2_processed")
    mf.log_param("scaler_type", "StandardScaler")
    mf.log_param("imputer_strategy", "median")
    
    joblib.dump(scaler, "scaler.pkl")
    mf.log_artifact("scaler.pkl")

    
    def evaluate_model(model, X_test, y_test):
        y_pred = model.predict(X_test)

        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        mf.log_metric("test_rmse", rmse)
        mf.log_metric("test_mae", mae)
        mf.log_metric("test_r2", r2)

        return rmse, mae, r2

    # Models:
    # Linear Regression
    with mf.start_run(run_name="Linear_SubRun", nested=True):
        lr = LinearRegression()
        lr.fit(X_train_scaled, y_train.values)

        evaluate_model(lr, X_test_scaled, y_test.values)

    # CART
    with mf.start_run(run_name="CART_SubRun", nested=True):
        dt = DecisionTreeRegressor(max_depth=5)
        dt.fit(X_train_scaled, y_train.values)

        #mf.log_param("max_depth", 5)

        evaluate_model(dt, X_test_scaled, y_test.values)

    # XGBoost
    with mf.start_run(run_name="XGBoost_SubRun", nested=True):
    
        xgb_model = XGBRegressor(
            n_estimators=100,
            learning_rate=0.1,
            random_state=42
        )

        xgb_model.fit(X_train_scaled, y_train.values)

        #mf.log_param("n_estimators", 100)
        #mf.log_param("learning_rate", 0.1)
        #mf.log_param("model_type", "XGBoost")

        evaluate_model(xgb_model, X_train_scaled, y_train.values)
        evaluate_model(xgb_model, X_test_scaled, y_test.values)

🏃 View run Linear_SubRun at: http://127.0.0.1:5000/#/experiments/1/runs/1b16039c3690418ebef7325bb8170657
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run CART_SubRun at: http://127.0.0.1:5000/#/experiments/1/runs/116790d6020748d3ba5767dc7b43f3dd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run XGBoost_SubRun at: http://127.0.0.1:5000/#/experiments/1/runs/f94537cea28f43379eea52c27b3853fd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Housing_Model_Comparison_v0 at: http://127.0.0.1:5000/#/experiments/1/runs/21d64f078c91447189f84037ed2a628f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
